# Privacy & PII: wrangling data about people responsibly

_Data Wrangling_

**Strip the obvious identifiers, then realize that ZIP + birthdate + gender still names you — and fix that too.**

Most real datasets are about people: users, patients, customers, employees. The moment a
       row can be traced back to a human, you have a duty &mdash; legal and ethical &mdash; to handle it
       with care. PII (Personally Identifiable Information) is any data that identifies a specific
       person, on its own or combined with other data.

       PII comes in two flavors, and the second is the one people forget. Direct identifiers
       point straight at someone: full name, email, Social Security Number, phone number, account ID.
       Quasi-identifiers look harmless one at a time but re-identify a person when combined.
       The classic result: a person's 5-digit ZIP code, birthdate, and gender together uniquely
       identify about 87% of the U.S. population (Latanya Sweeney's study). None of those three is a
       name &mdash; yet together they are as good as one.

---

This notebook is a practice scaffold for the **Privacy & PII: wrangling data about people responsibly** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — pandas

We take a customer table and de-identify it in four steps: (1) drop the direct identifiers, (2) replace the ID we must keep with a salted hash, (3) generalise the quasi-identifiers, and (4) check k-anonymity to see whether anyone is still uniquely identifiable.

### Step 1 — Drop the direct identifiersDirect identifiers point straight at a person — name, email, SSN. They have no analytic value and must go first. We build the synthetic table, then drop those columns (redacting in place is the alternative shown in the comment).

In [ ]:
import hashlib
import pandas as pd

# A small synthetic customer table (replace with your real frame).
df = pd.DataFrame({
    "user_id":  ["u-1001", "u-1002", "u-1003", "u-1004", "u-1005"],
    "name":     ["Ada Lovelace", "Alan Turing", "Grace Hopper", "Ada Byron", "Edsger D."],
    "email":    ["ada@x.com", "alan@x.com", "grace@x.com", "ada2@x.com", "ed@x.com"],
    "ssn":      ["111-22-3333", "222-33-4444", "333-44-5555", "444-55-6666", "555-66-7777"],
    "birthdate":["1983-07-14", "1991-02-03", "1983-11-30", "1991-08-22", "1955-01-09"],
    "zip":      ["02139", "02141", "02139", "02141", "30301"],
    "gender":   ["F", "M", "F", "M", "F"],
    "spend":    [120, 80, 200, 50, 300],
})

# Remove / redact DIRECT identifiers.
DIRECT_PII = ["name", "email", "ssn"]
df = df.drop(columns=DIRECT_PII)        # or: df[DIRECT_PII] = "[REDACTED]"


### Step 2 — Pseudonymise the ID we must keepWe still need a stable per-user key to link a person's rows, but we must not store the real ID. A **salted** SHA-256 hash gives a stable pseudonymous token: the salt is a secret that never ships with the data, which is what defeats a dictionary attack on guessable IDs.

In [ ]:
# Salted hash of the ID we must keep (links rows, hides the real id).
SALT = "k7$rotate-me-and-keep-me-secret"   # NEVER ship the salt with the data

def salted_hash(value, salt=SALT):
    salted = salt + str(value)
    digest = hashlib.sha256(salted.encode()).hexdigest()
    return digest[:16]

df["user_id"] = df["user_id"].map(salted_hash)   # -> stable pseudonymous token


### Step 3 — Generalise the quasi-identifiersBirthdate, ZIP, and gender are quasi-identifiers: harmless alone, but together they re-identify ~87% of people. We coarsen them — birthdate becomes a 10-year age band, the 5-digit ZIP becomes its first 3 digits — then drop the precise originals so the fingerprints are blunted.

In [ ]:
# Generalize the QUASI-identifiers.
birth_year = pd.to_datetime(df["birthdate"]).dt.year
age = 2026 - birth_year

df["age_band"] = (age // 10 * 10).astype(str) + "s"   # 41 -> "40s"
df["zip3"]     = df["zip"].str[:3]                     # "02139" -> "021"

df = df.drop(columns=["birthdate", "zip"])


### Step 4 — Check k-anonymityk-anonymity asks: for the quasi-identifiers, does every row sit in a group of at least `k` identical rows? We group on the generalised columns, attach each row's group size, and flag any row whose group is smaller than `k` — those people are still re-identifiable and would need suppression or further coarsening.

In [ ]:
# CHECK k-anonymity on the quasi-identifiers.
QUASI = ["age_band", "zip3", "gender"]
K = 2

group_sizes = df.groupby(QUASI).transform("size")    # size of each row's group
df["group_size"] = group_sizes

violations = df[df["group_size"] < K]                # rows in too-small groups

print("min group size:", int(df["group_size"].min()),
      "| k-anonymous for k =", K, ":", df["group_size"].min() >= K)
print("re-identifiable rows (group <", K, "):", len(violations))
# Fix violations by suppressing those rows or generalizing further (e.g. drop gender).


## Visualize the data & results

_How many people are uniquely identifiable (group size 1) before vs after we generalize the quasi-identifiers? Group the synthetic table on Q = (birthdate/age, ZIP, gender) and count the group sizes._

### Step 1 — Build an 8-person table with exact quasi-identifiersTo see generalisation work, we need rows that are unique on their precise quasi-identifiers. This synthetic table gives every person a distinct (birthdate, ZIP, gender) triple — the raw fingerprints we are about to blunt.

In [ ]:
import pandas as pd

# 8-person synthetic table (exact quasi-identifiers).
df = pd.DataFrame({
    "birthdate": ["1983-07-14","1991-02-03","1983-11-30","1991-08-22",
                  "1955-01-09","1988-05-05","1992-09-19","1986-12-01"],
    "zip":       ["02139","02141","02140","02142","30301","02143","02144","02145"],
    "gender":    ["F","M","F","M","F","F","M","F"],
})


### Step 2 — Count group sizes BEFORE generalisingGrouping on the exact birthdate, 5-digit ZIP, and gender, every person lands in a group of one. All group sizes are 1, so the table is only 1-anonymous — fully re-identifiable by linkage to outside data.

In [ ]:
# BEFORE: group on exact birthdate + 5-digit ZIP + gender.
before = df.groupby(["birthdate","zip","gender"]).size()

print("BEFORE group sizes:", sorted(before.values, reverse=True))
# -> [1, 1, 1, 1, 1, 1, 1, 1]   (8 unique fingerprints -> 1-anonymous)


### Step 3 — Generalise, then count againWe coarsen birthdate to an age band and ZIP to its first 3 digits, then re-group. Some singletons now merge into groups of 3, but a couple remain alone — the minimum group size is still 1, so the data is not yet 2-anonymous and those stragglers need more work.

In [ ]:
# Generalize: birthdate -> age-band, ZIP-5 -> ZIP-3.
age = 2026 - pd.to_datetime(df["birthdate"]).dt.year
df["age_band"] = (age // 10 * 10).astype(str) + "s"
df["zip3"]     = df["zip"].str[:3]

# AFTER: group on the generalized quasi-identifiers.
after = df.groupby(["age_band","zip3","gender"]).size()

print("AFTER group sizes :", sorted(after.values, reverse=True))
# -> [3, 3, 1, 1]   (singletons collapse; min group = 1, so not yet 2-anonymous)


## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A colleague hands you a "fully anonymized" customer table: they deleted name and email, but kept exact birthdate, full 5-digit zip, and gender. Is it safe to publish? Justify with the concept and give the fix.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Identify the surviving fields as quasi-identifiers, not harmless columns. — _Birthdate, ZIP, and gender together re-identify ~87% of people even with no name present._
- Group the table by (birthdate, zip, gender) and inspect group sizes. — _Most combinations will be unique (size 1), meaning the data is only 1-anonymous &mdash; fully re-identifiable by linkage._
- Generalize: birthdate &rarr; age-band, zip-5 &rarr; zip-3, then re-group and confirm every group has &ge; k members; suppress or coarsen the stragglers. — _Merging rare fingerprints into groups of size &ge; k is exactly what makes the data k-anonymous._

**Answer:** No. Deleting names removed only the direct identifiers; birthdate + ZIP + gender are quasi-identifiers that re-identify people by linking to outside data. Grouped on those three, almost every row is unique (1-anonymous). Fix: generalize &mdash; age-band instead of birthdate, ZIP-3 instead of ZIP-5 &mdash; then verify $k$-anonymity (smallest group $\ge k$), suppressing or further coarsening any leftover singletons.

</details>

**Problem 2.** You need to keep a per-user key so you can count sessions per person, but you must not store real user emails. A teammate proposes sha256(email). Why is that still unsafe, and what do you change?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note the email space is enumerable / guessable. — _An attacker can hash a huge dictionary of known or common emails and match the digests &mdash; an unsalted hash is effectively reversible._
- Add a long secret salt kept out of the dataset: hash(salt + email). — _The salt is unknown to the attacker, so a precomputed dictionary or rainbow table no longer matches._
- Prefer a slow/keyed hash and rotate or vault the salt. — _Slows brute force and limits blast radius if a salt leaks; consistent salt still gives a stable per-user token for counting._

**Answer:** Plain sha256(email) is reversible by a dictionary attack: emails are guessable, so an attacker hashes them all and matches. Mix in a secret salt first &mdash; sha256(salt + email) &mdash; keep the salt out of the shared data, and prefer a slow/keyed hash. You still get a stable token to count sessions, but it can't be reversed.

</details>

**Problem 3.** Your stakeholder only needs "average order value by region and age-band", not row-level data. How does that change your privacy strategy, and what is the one remaining trap?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize that the question is answerable from aggregates, not individual rows. — _Releasing counts/means per group reveals no single person, the strongest and simplest protection._
- Aggregate to (region, age-band) &rarr; mean order value, and drop the row-level table from the export. — _No quasi-identifier fingerprints leave the building, so there is nothing to re-identify._
- Check that each group is large enough (and consider adding noise) before publishing. — _A group of one person makes the "average" that person's exact value &mdash; an aggregate can still leak; differential privacy adds calibrated noise to close this gap._

**Answer:** Aggregate. Since only group-level facts are needed, publish counts and means per (region, age-band) and never export the rows &mdash; the safest option. The remaining trap: tiny groups. An average over one person is that person's value, so enforce a minimum group size and, for strong guarantees, add differential-privacy noise to the aggregates.

</details>